# FEMI-4 — Causal decomposition of a verification-interface failure

**Locally frozen 576-call protocol.** No API calls occur until you explicitly run the **LIVE COLLECTION** cell. This notebook retains the exact original FEMI-2 messages in reference conditions and manipulates source-boundary safeguards and upstream verification claims separately. Read `FEMI4_PROTOCOL_LOCALLY_FROZEN.md` before the live run.

**Security:** Use a newly rotated OpenRouter key in Colab Secrets under `OPENROUTER_API_KEY`; never paste a key into a notebook cell, terminal output or chat. Groq-only routing is enforced and fallback is disabled.

## 1. Upload and extract the complete experimental ZIP

Upload `FEMI4_CausalInterface_20261108.zip` rather than individual files. The notebook will not overwrite an existing results directory.

In [ ]:
from pathlib import Path
from google.colab import files
import zipfile, subprocess, os, json
uploaded = files.upload()
candidates = [name for name in uploaded if name.endswith('.zip') and name.startswith('FEMI4_CausalInterface_')]
assert len(candidates) == 1, 'Upload exactly one FEMI-4 experiment ZIP.'
with zipfile.ZipFile(candidates[0]) as archive:
    bad = archive.testzip()
    assert bad is None, f'Bad ZIP member: {bad}'
    archive.extractall('/content')
P = Path('/content/FEMI4_CausalInterface_20261108')
assert (P/'femi4.py').exists() and (P/'offline/femi4_freeze.json').exists()
print('Study directory:', P)

## 2. OFFLINE PREFLIGHT — no paid inference

The tests cover exact source requests, all intervention slots, record presence, provider configuration, hashes, missing/duplicate response rejection, and a **synthetic fixture** used only to test the analyzer. These are not research results.

In [ ]:
subprocess.run(['python','-m','unittest','discover','-s','tests','-v'],cwd=P,check=True)
subprocess.run(['python','femi4.py','--preflight'],cwd=P,check=True)
print('PRE-FLIGHT COMPLETE: zero model calls made.')

## 3. LIVE COLLECTION — makes API requests

Only run after the preflight passes. For a private key, open the Colab left-sidebar **Secrets** tab and set `OPENROUTER_API_KEY`, granting notebook access. Run is resumable if it is interrupted, provided the `femi4_results/` folder remains available. Calls are in a frozen randomized order. Don't inspect partial outcomes to change the plan.

In [ ]:
from google.colab import userdata
secret = userdata.get('OPENROUTER_API_KEY')
assert isinstance(secret,str) and secret.strip(), 'Set OPENROUTER_API_KEY in Colab Secrets first.'
os.environ['OPENROUTER_API_KEY'] = secret
# Never print secret. Restrict this notebook and rotate any previously exposed key.
del secret
subprocess.run(['python','femi4.py','--live','--out','femi4_results','--pace','0.85'],cwd=P,check=True)
print('Live collection step finished. The next cell will verify completion before analysis.')

## 4. ANALYZE — requires all 576 valid responses

The analyzer recomputes raw JSON verdicts, validates the complete response inventory and reports all 16 factorial cells. A rationale review sheet is provided with **blank semantic labels**; don't conflate structural citations with verified false attribution.

In [ ]:
subprocess.run(['python','femi4.py','--analyze','--out','femi4_results'],cwd=P,check=True)
result=json.loads((P/'femi4_results'/'analysis'/'analysis.json').read_text())
print('Completed calls:',result['n_calls'])
print('12 case-cluster primary contrast:',result['contrasts']['primary_guard_under_original_promise_handoff'])

## 5. EXPORT

Download the archive and upload it to the conversation for an independent audit. Export includes frozen manifest hashes, raw responses, attempts and scored results. It does not contain your API key.

In [ ]:
subprocess.run(['python','femi4.py','--export','--out','femi4_results'],cwd=P,check=True)
files.download(str(P/'FEMI4_LIVE_RESULTS.zip'))